# SmartBiz AI — UCI Online Retail EDA
## Capstone Project: Demand Forecasting for Small Retail Businesses

**Dataset:** UCI Online Retail Dataset  
**Source:** https://archive.ics.uci.edu/dataset/352/online+retail  
**Goal:** Data wrangling

---
### Notebook Structure
1. Install & Import Libraries
2. Load Data
3. Quick Preview
4. Quick Data Cleaning
5. **ydata-profiling Report**
6. Remaining Data Cleaning
7. Outlier Analysis

---
## 1. Install & Import Libraries

In [3]:
# Install required libraries
!pip install ydata-profiling openpyxl pandas numpy matplotlib seaborn

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from ydata_profiling import ProfileReport
from IPython.display import display, HTML


---
## 2. Load Data

In [6]:
#FILE_PATH = '/Users/jacobpaul/Documents/Springboard-bootcamp/GitHub/retail-demand-forecasting-capstone/data/Online Retail.xlsx'
#df = pd.read_excel(FILE_PATH, engine='openpyxl')
#print(f'Loaded: {df.shape}')

Loaded: (541909, 8)


In [44]:
import pandas as pd

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
df = pd.read_excel(URL, engine='openpyxl')
print(f'Loaded: {df.shape}')

Loaded: (541909, 8)


---
## 3. Quick Preview

In [8]:
print('=== Shape ===')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')

print('\n=== First 5 Rows ===')
display(df.head())

print('\n=== Data Types ===')
display(df.dtypes.to_frame('dtype'))

print('\n=== Missing Values ===')
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
display(missing[missing['Missing Count'] > 0])

print('\n=== Basic Statistics ===')
display(df.describe())
raw_count = len(df)

=== Shape ===
Rows: 541,909  |  Columns: 8

=== First 5 Rows ===


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



=== Data Types ===


,dtype
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
UnitPrice,float64
CustomerID,float64
Country,object



=== Missing Values ===


,Missing Count,Missing %
Description,1454,0.27
CustomerID,135080,24.93



=== Basic Statistics ===


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


---
## 4. Quick Data Cleaning

In [10]:
print('\n=== Validating InvoiceNo and removing C and A ===')
print(df['InvoiceNo'].astype(str).str[0].value_counts())
print(df[df['InvoiceNo'].astype(str).str.startswith(('A','C'))])
df = df[~df['InvoiceNo'].astype(str).str.startswith(('A','C'))]


=== Validating InvoiceNo and removing C and A ===
InvoiceNo
5    532618
C      9288
A         3
Name: count, dtype: int64
       InvoiceNo StockCode                       Description  Quantity  \
141      C536379         D                          Discount        -1   
154      C536383    35004C   SET OF 3 COLOURED  FLYING DUCKS        -1   
235      C536391     22556    PLASTERS IN TIN CIRCUS PARADE        -12   
236      C536391     21984  PACK OF 12 PINK PAISLEY TISSUES        -24   
237      C536391     21983  PACK OF 12 BLUE PAISLEY TISSUES        -24   
...          ...       ...                               ...       ...   
540449   C581490     23144   ZINC T-LIGHT HOLDER STARS SMALL       -11   
541541   C581499         M                            Manual        -1   
541715   C581568     21258        VICTORIAN SEWING BOX LARGE        -5   
541716   C581569     84978  HANGING HEART JAR T-LIGHT HOLDER        -1   
541717   C581569     20979     36 PENCILS TUBE RED RETROSPOT   

In [11]:
print('\n=== Validating UnitPrice and Remove Zero ===')
df['UnitPrice'].value_counts()
print(df[df['UnitPrice']<0])
df = df[df['UnitPrice']>0]


=== Validating UnitPrice and Remove Zero ===
Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country]
Index: []


In [12]:
print('\n=== Validating Quantity and keeps only positive qty ===')
print(df['Quantity'].value_counts())
(df['Quantity']==0).sum()
df = df[df['Quantity']>0]


=== Validating Quantity and keeps only positive qty ===
Quantity
1        147822
2         81699
12        61044
6         40839
4         38442
          ...  
828           1
560           1
512           1
291           1
80995         1
Name: count, Length: 375, dtype: int64


In [13]:
print('\n=== Find Duplicates and drop ===')
print(f' Duplicates found: {df.duplicated().sum()}')
df = df.drop_duplicates()


=== Find Duplicates and drop ===
 Duplicates found: 5226


In [14]:
print('\n=== Summary ===')
print(f'Missing Description rows: {df["Description"].isna().sum()}')
print(f'Date range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
print(f'Unique customers: {df["CustomerID"].nunique():,}')
print(f'Unique products:  {df["StockCode"].nunique():,}')
print(f'Unique countries: {df["Country"].nunique():,}')
print(f'Unique invoices:  {df["InvoiceNo"].nunique():,}')
print(f'Retention rate: {len(df)/raw_count*100:.1f}%')


=== Summary ===
Missing Description rows: 0
Date range: 2010-12-01 08:26:00 → 2011-12-09 12:50:00
Unique customers: 4,338
Unique products:  3,921
Unique countries: 38
Unique invoices:  19,959
Retention rate: 96.9%


---
## 5. ydata-profiling Report




In [16]:
print('\n=== Ydata Report generation ===')
profile = ProfileReport(
    df,
    title='SmartBiz AI — UCI Online Retail EDA Report',
    explorative=True,
    correlations={
        "pearson":  {"calculate": True},
        "spearman": {"calculate": True},
        "kendall":  {"calculate": False},
        "phi_k":    {"calculate": False},
        "cramers":  {"calculate": False},
    }
)

# Display inline
profile.to_notebook_iframe()
#Saving html to local
profile.to_file('/Users/jacobpaul/Documents/Springboard-bootcamp/GitHub/retail-demand-forecasting-capstone/reports/ydata_eda_profile.html')


=== Ydata Report generation ===


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|█████████████████████████████████████████████| 8/8 [00:00<00:00,  9.51it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

---
## 6. Remaining Data Cleaning

Based on the profile report, we apply the following cleaning steps:

In [18]:
# Adding useful columns
df['Revenue']    = df['Quantity'] * df['UnitPrice']
df['Date']       = pd.to_datetime(df['InvoiceDate'].dt.date)
df['Month']      = df['InvoiceDate'].dt.to_period('M').astype(str)
df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()
df['Hour']       = df['InvoiceDate'].dt.hour
df['Week']       = df['InvoiceDate'].dt.isocalendar().week.astype(int)
df['MonthNum']   = df['InvoiceDate'].dt.month
df['DayNum']     = df['InvoiceDate'].dt.dayofweek

print(f'\n Final clean dataset: {len(df):,} rows')
print(f'Retention rate: {len(df)/raw_count*100:.1f}%')
print(f'\nNote: CustomerID still missing for {df["CustomerID"].isna().sum():,} rows')


 Final clean dataset: 524,877 rows
Retention rate: 96.9%

Note: CustomerID still missing for 132,185 rows


### Missing Value Strategy
- **CustomerID** (135K missing, ~25%): Retained intentionally.
  All rows are used for demand forecasting since CustomerID is not 
  a model feature. Only rows with CustomerID are used for RFM/churn analysis.

In [20]:
# Subset with CustomerID for churn analysis
df_cust = df[df['CustomerID'].notna()].copy()
print(f'Rows with CustomerID: {len(df_cust):,}')
print(f'Unique customers: {df_cust["CustomerID"].nunique():,}')
display(df.head())

Rows with CustomerID: 392,692
Unique customers: 4,338


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Date,Month,DayOfWeek,Hour,Week,MonthNum,DayNum
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12-01,2010-12,Wednesday,8,48,12,2
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,2010-12,Wednesday,8,48,12,2
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12-01,2010-12,Wednesday,8,48,12,2
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,2010-12,Wednesday,8,48,12,2
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12-01,2010-12,Wednesday,8,48,12,2


---
## 7. Outlier Analysis

In [22]:
print('=== Outlier Analysis ===')

# Quantity outliers
q99 = df['Quantity'].quantile(0.99)
q_outliers = (df['Quantity'] > q99).sum()
print(f'Quantity 99th percentile: {q99}')
print(f'Rows above 99th pct: {q_outliers}')

# UnitPrice outliers
p99 = df['UnitPrice'].quantile(0.99)
p_outliers = (df['UnitPrice'] > p99).sum()
print(f'UnitPrice 99th percentile: {p99:.2f}')
print(f'Rows above 99th pct: {p_outliers}')

# Revenue outliers
r99 = df['Revenue'].quantile(0.99)
r_outliers = (df['Revenue'] > r99).sum()
print(f'Revenue 99th percentile: {r99:.2f}')
print(f'Rows above 99th pct: {r_outliers}')

=== Outlier Analysis ===
Quantity 99th percentile: 100.0
Rows above 99th pct: 4847
UnitPrice 99th percentile: 16.98
Rows above 99th pct: 5073
Revenue 99th percentile: 183.60
Rows above 99th pct: 5231


Outliers above the 99th percentile are retained in the wrangling stage. They will be capped during feature engineering to prevent extreme values from distorting the XGBoost model.